# Constrained RNN Paradigm A

This notebook demonstrates the **ConstrainedRNN** family in NeuralRNN:

1. **Spatially-embedded RNN (seRNN)** on a maze first-choice task, using the NeuralRNN training pipeline.
2. **SparseRNN vs. CTRNN** on the context-dependent decision-making (Mante) task, comparing different recurrent sparsity levels (5%, 10%, 50%).
3. **SparseRNN vs. CTRNN** on the DelayComparison (parametric working memory) task, comparing delay-period geometry and line-attractor structure.

Reference projects:

In [ ]:
import sys
sys.path.append('../src')

import os
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
from scipy.spatial import distance
import scipy.stats

from neuralrnn import AutoConfig, AutoModel, Trainer, TrainingArguments, load_dataset
from neuralrnn import SupervisedObjective
from neuralrnn.train.objectives.constrained import ConstrainedSupervisedObjective
from neuralrnn.analysis import fit_pca, find_fixed_points
from neuralrnn.analysis.linearization import linearize, dominant_direction
from neuralrnn.data.base import BaseDataset

# Reproducibility
np.random.seed(1829)
torch.manual_seed(9492)

print('torch:', torch.__version__)
print('numpy:', np.__version__)

# Checkpoint directory
MODEL_DIR = Path('./models/10')
MODEL_DIR.mkdir(parents=True, exist_ok=True)

## Part I: Spatially-embedded RNN (seRNN) Demo

This section reproduces the maze first-choice demo from Achterberg et al. (2023), *Spatially embedded recurrent neural networks reveal widespread links between structural and functional neuroscience findings* (`reference_project/constrained_rnn/spatially-embedded-RNN/`).

### Task and data

The network observes a one-step navigation problem:
1. **Goal presentation** (20 steps): a cue indicates one of four goal positions on a grid (top/bottom × left/right).
2. **Delay period** (10 steps): the goal location must be maintained in memory.
3. **Choice presentation** (20 steps): two possible movement directions are shown.
4. The network must report the direction of the first step that moves closer to the goal.

Both observations and labels are one-hot encoded. The input has 8 channels (4 goal positions + 4 choice directions); the output has 4 channels (left/up/right/down). Gaussian noise with standard deviation 0.05 is added to the inputs, matching the original paper.

### Model

The network is a 100-unit rate RNN placed on a regular $5 \times 5 \times 4$ grid. Its discrete-time dynamics are

$$
h_t = \mathrm{ReLU}\left(W_{\mathrm{rec}} h_{t-1} + W_{\mathrm{in}} x_t + b_{\mathrm{rec}}\right), \qquad
y_t = W_{\mathrm{out}} h_t + b_{\mathrm{out}}.
$$

The recurrent weights are initialized orthogonally; input and readout weights use Xavier initialization.

### Spatial regularizer

seRNNs add a structural penalty that favors short, topologically central connections:

$$
\mathcal{L}_{\mathrm{struct}} = \lambda \sum_{ij} |W^{\mathrm{rec}}_{ij}| \; d_{ij}^{\,p} \; C_{ij}^{\,\alpha},
$$

where
- $d_{ij}$ is the Euclidean distance between units $i$ and $j$ on the 3-D grid,
- $C = \exp\left(S^{-1/2} |W^{\mathrm{rec}}| S^{-1/2}\right)$ is the unbiased weighted communicability matrix, with $S_{ii} = \sum_j |W^{\mathrm{rec}}_{ij}|$ the node strength,
- $\lambda$ (`se1_weight`) and $\alpha$ (`comms_factor`) control the regularization strength.

The full training objective is the cross-entropy loss plus the structural loss:

$$
\mathcal{L} = \mathcal{L}_{\mathrm{task}} + \mathcal{L}_{\mathrm{struct}}.
$$

### Training protocol

Following the paper Methods: Adam optimizer, learning rate $10^{-3}$, $\beta_1=0.9$, $\beta_2=0.999$, $\epsilon=10^{-7}$, batch size 128, 10 epochs of 5,120 problems each.

In [ ]:
class MazeGeneratorI:
    """First-choice maze task generator (numpy / PyTorch version of seRNN demo)."""

    def __init__(self, goal_presentation_steps=20, delay_steps=10, choices_presentation_steps=20):
        self.goal_presentation_steps = goal_presentation_steps
        self.delay_steps = delay_steps
        self.choices_presentation_steps = choices_presentation_steps
        self.mazesdf = self._import_maze_dic()
        self.mazesdf['Goal_Presentation'] = self.mazesdf['goal'].map({
            7: np.concatenate((np.array([1, 0, 0, 0]), np.repeat(0, 4))),
            9: np.concatenate((np.array([0, 1, 0, 0]), np.repeat(0, 4))),
            17: np.concatenate((np.array([0, 0, 1, 0]), np.repeat(0, 4))),
            19: np.concatenate((np.array([0, 0, 0, 1]), np.repeat(0, 4))),
        })
        self.mazesdf['Choices_Presentation'] = self.mazesdf['ChoicesCategory'].map(
            lambda x: self._encode_choices(x)
        )
        self.mazesdf['Step_Encoded'] = self.mazesdf['NextFPmap'].map(
            lambda x: self._encode_next_step(x)
        )

    def _import_maze_dic(self):
        mazesDic = {
            'goal': {0: 9, 1: 9, 2: 19, 3: 17, 4: 17, 5: 7, 6: 19, 7: 7},
            'ChoicesCategory': {0: 'ul', 1: 'rd', 2: 'ld', 3: 'rd', 4: 'ul', 5: 'ur', 6: 'lr', 7: 'lr'},
            'NextFPmap': {0: 'u', 1: 'r', 2: 'd', 3: 'd', 4: 'l', 5: 'u', 6: 'r', 7: 'l'},
        }
        return pd.DataFrame(mazesDic)

    @staticmethod
    def _encode_choices(x):
        choices_sec = np.repeat(0, 4)
        choicesEncoding = pd.Series(list(x))
        choicesEncoding = choicesEncoding.map({'l': 1, 'u': 2, 'r': 3, 'd': 4})
        for encodedChoice in choicesEncoding:
            choices_sec[encodedChoice - 1] = 1
        return np.concatenate((np.repeat(0, 4), choices_sec))

    @staticmethod
    def _encode_next_step(x):
        step_sec = np.repeat(0, 4)
        stepEncoding = pd.Series(list(x))
        stepEncoding = stepEncoding.map({'l': 1, 'u': 2, 'r': 3, 'd': 4})
        for encodedStep in stepEncoding:
            step_sec[encodedStep - 1] = 1
        return step_sec

    def _create_problem_observation(self, row):
        goal_vec = np.tile(row['Goal_Presentation'], self.goal_presentation_steps)
        delay_vec = np.tile(np.repeat(0, 8), self.delay_steps)
        choices_vec = np.tile(row['Choices_Presentation'], self.choices_presentation_steps)
        return np.concatenate((goal_vec, delay_vec, choices_vec))

    def construct_numpy_data(self, number_of_problems, shuffle_labels=False):
        self.mazesdf['Problem_Vec'] = self.mazesdf.apply(
            lambda row: self._create_problem_observation(row), axis=1
        )
        self.mazes_order = np.random.randint(0, 8, number_of_problems)

        session_observation = np.array([])
        session_labels = np.array([])
        for i in self.mazes_order:
            session_observation = np.append(session_observation, self.mazesdf.iloc[i]['Problem_Vec'])
            session_labels = np.append(session_labels, self.mazesdf.iloc[i]['Step_Encoded'])

        session_length = (self.goal_presentation_steps + self.delay_steps +
                          self.choices_presentation_steps)
        session_observation = np.reshape(session_observation, (-1, session_length, 8)).astype('float32')
        session_labels = np.reshape(session_labels, (-1, 4)).astype('float32')

        if shuffle_labels:
            shuffle_generator = np.random.default_rng(38446)
            shuffle_generator.shuffle(session_labels, axis=0)
        return session_observation, session_labels


class MazeDataset(BaseDataset):
    """NeuralRNN-compatible dataset for the seRNN maze task.

    Targets are class indices with a per-timestep mask: only the final timestep
    contributes to the cross-entropy loss, because the network response is read
    out once at the end of the trial.
    """

    kind = "supervised"

    def __init__(self, observations, labels, batch_size, add_noise=True):
        super().__init__()
        self.observations = torch.from_numpy(observations).float()
        self.labels = torch.from_numpy(labels).float()
        self.batch_size = batch_size
        self.add_noise = add_noise
        self.training = True

        n_trials, trial_len, input_dim = self.observations.shape
        self.input_dim = input_dim
        self.output_dim = self.labels.shape[-1]

        # Targets: class index only at final timestep; -100 is ignored by the mask
        self.targets = torch.full((n_trials, trial_len), -100, dtype=torch.long)
        self.targets[:, -1] = torch.argmax(self.labels, dim=1)
        self.mask = torch.zeros(n_trials, trial_len, dtype=torch.float32)
        self.mask[:, -1] = 1.0

    def sample_batch(self):
        idx = torch.randint(0, len(self.observations), (self.batch_size,))
        x = self.observations[idx].clone()
        if self.add_noise and self.training:
            x = x + torch.randn_like(x) * 0.05
        return {
            "inputs": x,
            "targets": self.targets[idx],
            "mask": self.mask[idx],
        }

    def __len__(self):
        return self.observations.shape[0]


gen = MazeGeneratorI(goal_presentation_steps=20, delay_steps=10, choices_presentation_steps=20)
train_obs, train_labels = gen.construct_numpy_data(number_of_problems=5120)
val_obs, val_labels = gen.construct_numpy_data(number_of_problems=2560)

train_ds = MazeDataset(train_obs, train_labels, batch_size=128, add_noise=True)
val_ds = MazeDataset(val_obs, val_labels, batch_size=128, add_noise=False)
print('Train trials:', len(train_ds), '| Val trials:', len(val_ds))
print('Input shape:', train_ds.observations.shape)
print('Target shape:', train_ds.targets.shape)

In [ ]:
# Build seRNN: 100 hidden units on a 5x5x4 3D grid
cfg_se = AutoConfig.for_model(
    'se_rnn',
    input_dim=8,
    latent_dim=100,
    output_dim=4,
    dt=None,              # discrete vanilla RNN style (alpha = 1)
    tau=1.0,
    grid_shape=(5, 5, 4),
    se1_weight=0.5,
    comms_factor=1.0,
    distance_power=1.0,
    distance_metric='euclidean',
    orthogonal_init=True,
    activation='relu',
)
model_se = AutoModel.from_config(cfg_se)
print(model_se)
print('Distance matrix shape:', model_se.distance_matrix.shape)
print('Coordinates shape:', model_se.coordinates.shape)

In [ ]:
SAVE_SE = MODEL_DIR / 'se_rnn'

if SAVE_SE.exists():
    model_se = AutoModel.from_pretrained(SAVE_SE)
    print('Loaded pretrained seRNN from', SAVE_SE)
else:
    objective = ConstrainedSupervisedObjective(
        task_type='classification', constraint_weight=1.0
    )
    args = TrainingArguments(
        max_steps=400,        # 10 epochs * (5120 / 128) batches
        learning_rate=1e-3,
        batch_size=128,
        log_every=40,
        seed=1829,
        grad_clip_norm=1.0,
    )

    history = Trainer(model_se, train_ds, objective, args).train()

    # Validation pass
    model_se.eval()
    val_ds.training = False
    val_correct, val_total = 0, 0
    with torch.no_grad():
        for _ in range(len(val_ds) // val_ds.batch_size):
            batch = val_ds.sample_batch()
            out = model_se(batch['inputs'])
            logits = out.outputs[:, -1, :]
            preds = torch.argmax(logits, dim=1)
            val_correct += (preds == batch['targets'][:, -1]).sum().item()
            val_total += preds.size(0)
    val_acc = val_correct / val_total
    print(f'Validation accuracy: {val_acc:.4f}')

    model_se.save_pretrained(SAVE_SE)
    print('Saved seRNN to', SAVE_SE)

In [ ]:
# Collect recurrent weight history for structural analysis
weight_history = [model_se.h2h.weight.detach().cpu().clone().numpy().copy()]

# If we loaded a pretrained model, we do not have per-epoch history; just analyse final.
example_weight_matrix = np.abs(weight_history[-1])

# Binarize (top 10% of absolute weights)
binary_weight_matrix = example_weight_matrix.copy()
thresh = np.quantile(example_weight_matrix, q=0.9)
binary_weight_matrix[example_weight_matrix > thresh] = 1.0
binary_weight_matrix[example_weight_matrix <= thresh] = 0.0


def modularity_und(binary_weight_matrix, gamma=1.0, seed=0):
    """Manual modularity calculation using a simple Louvain-like procedure."""
    rng = np.random.default_rng(seed)
    A = np.array(binary_weight_matrix, dtype=np.float64)
    A = (A + A.T) / 2.0
    A = (A > 0).astype(np.float64)
    n = A.shape[0]
    m = np.sum(A) / 2.0
    if m == 0:
        return np.arange(n), 0.0
    degrees = np.sum(A, axis=1)
    communities = np.arange(n)
    improved = True
    while improved:
        improved = False
        for node in rng.permutation(n):
            current_comm = communities[node]
            neighbors = np.where(A[node] > 0)[0]
            neighbor_comms = set(communities[neighbors])
            best_comm, best_gain = current_comm, 0.0
            for new_comm in neighbor_comms:
                if new_comm == current_comm:
                    continue
                gain = 0.0
                for j in range(n):
                    if j == node:
                        continue
                    if communities[j] == new_comm:
                        gain += 2 * (A[node, j] - gamma * degrees[node] * degrees[j] / (2 * m))
                    elif communities[j] == current_comm:
                        gain -= 2 * (A[node, j] - gamma * degrees[node] * degrees[j] / (2 * m))
                if gain > best_gain:
                    best_gain = gain
                    best_comm = new_comm
            if best_gain > 0:
                communities[node] = best_comm
                improved = True
    Q = 0.0
    for i in range(n):
        for j in range(n):
            if communities[i] == communities[j]:
                Q += A[i, j] - gamma * (degrees[i] * degrees[j]) / (2 * m)
    Q = Q / (2 * m)
    return communities, Q


_, q_stat = modularity_und(binary_weight_matrix)
print('Modularity:', q_stat)

try:
    import bct
    A = binary_weight_matrix
    clu = np.mean(bct.clustering_coef_bu(A))
    pth = bct.efficiency_bin(A)
    nperm = 1000
    cluperm = np.zeros(nperm)
    pthperm = np.zeros(nperm)
    rng = np.random.default_rng(0)
    for perm in range(nperm):
        Wperm = rng.random((100, 100))
        Wperm = (Wperm + Wperm.T) / 2.0
        Aperm = (Wperm > 0.7).astype(float)
        cluperm[perm] = np.mean(bct.clustering_coef_bu(Aperm))
        pthperm[perm] = bct.efficiency_bin(Aperm)
    smw = (clu / cluperm.mean()) / (pthperm.mean() / pth)
    print('Small-worldness:', smw)
except Exception as e:
    print('bctpy not installed; small-worldness analysis skipped:', e)
    smw = None

In [ ]:
fig = plt.figure(figsize=(12, 4))

# Distance matrix
ax1 = fig.add_subplot(1, 3, 1)
im1 = ax1.imshow(model_se.distance_matrix.cpu().numpy())
ax1.set_title('Distance matrix')
fig.colorbar(im1, ax=ax1)

# Recurrent weight matrix
ax2 = fig.add_subplot(1, 3, 2)
im2 = ax2.imshow(example_weight_matrix, aspect='auto')
ax2.set_title('Final recurrent weight matrix (|W|)')
fig.colorbar(im2, ax=ax2)

# 3-D neuron positions
ax3 = fig.add_subplot(1, 3, 3, projection='3d')
coords = model_se.get_neuron_positions()
ax3.scatter(coords[:, 0], coords[:, 1], coords[:, 2], c='b', marker='.')
ax3.set_xlabel('x')
ax3.set_ylabel('y')
ax3.set_zlabel('z')
ax3.set_title('Neuron positions in 3D space')

plt.tight_layout()
plt.show()

# Weight-distance correlation
flat_dist = model_se.distance_matrix.cpu().numpy().flatten()
corr = np.corrcoef(flat_dist, example_weight_matrix.flatten())[0, 1]
print(f'Weight-distance correlation: {corr:.4f}')
print(f'Total recurrent weight: {example_weight_matrix.sum():.2f}')

## Part II: SparseRNN vs. CTRNN on Context-Dependent Decision Making (Mante)

We compare a baseline CTRNN with SparseRNNs at 5%, 10%, and 50% recurrent connectivity on the Mante task (Mante et al., 2013, *Nature*).

### Task

On each trial the network receives six inputs:
- Channels 0–1: context cue (`motion` or `color`, amplitude 1.2).
- Channels 2–3: motion evidence (right / left).
- Channels 4–5: color evidence (right / left).

The task is to report the stronger motion coherence when the context is motion, and the stronger color coherence when the context is color. The context cue precedes the stimulus, and the network responds during a short decision window. Correct responses are encoded by high activity on output channel 0 and low activity on channel 1 (or vice versa).

### Model

All networks have 100 hidden units. The baseline CTRNN is fully connected; SparseRNNs keep only a fraction $s$ of recurrent connections:

$$
M_{ij} \sim \mathrm{Bernoulli}(s), \qquad W^{\mathrm{eff}}_{ij} = M_{ij} \, W^{\mathrm{raw}}_{ij}.
$$

Because the mask is registered as a non-trainable buffer and multiplied with the weight at every forward pass, masked weights stay zero and receive no gradient.

### Training

We use masked mean-squared error on the two readout channels. The task is trained with `dt=20` ms, `tau=100` ms (`\alpha = 0.2`) and `tanh` activation, which is stable for context-dependent integration in this framework.

In [ ]:
# Load Mante task
train_ds_mante = load_dataset('mante', batch_size=128, n_trials=40, n_coh=6)
test_ds_mante = load_dataset('mante', batch_size=128, n_trials=20, n_coh=6)

print('Train inputs:', train_ds_mante.inputs.shape)
print('Train targets:', train_ds_mante.targets.shape)
print('Train mask:', train_ds_mante.mask.shape)
print('Input dim:', train_ds_mante.input_dim, 'Output dim:', train_ds_mante.output_dim)
print('Example condition:', test_ds_mante.conditions[0])

In [ ]:
# Visualize a few example trials
fig, axes = plt.subplots(8, 1, figsize=(8, 10), sharex=True)

channel_names = ['ctx motion', 'ctx color', 'motion R', 'motion L', 'color R', 'color L', 'target 0', 'target 1']
for trial_idx in range(3):
    inp = test_ds_mante.inputs[trial_idx].numpy()
    tgt = test_ds_mante.targets[trial_idx].numpy()
    for ch in range(6):
        axes[ch].plot(inp[:, ch], alpha=0.7, label=f'trial {trial_idx}')
    for ch in range(2):
        axes[6 + ch].plot(tgt[:, ch], alpha=0.7)

for ch, name in enumerate(channel_names):
    axes[ch].set_ylabel(name)
axes[-1].set_xlabel('Time step')
axes[0].legend(loc='upper right', fontsize=7)
fig.suptitle('Mante task: example inputs and targets', fontsize=10)
plt.tight_layout()
plt.show()

In [ ]:
def train_mante(model, dataset, max_steps=5000, lr=1e-3):
    """Train a model on the Mante task and return history."""
    def _post_step_hook(m):
        if hasattr(m, '_apply_masks_to_weights'):
            m._apply_masks_to_weights()

    objective = SupervisedObjective(task_type='regression')
    args = TrainingArguments(
        max_steps=max_steps,
        learning_rate=lr,
        grad_clip_norm=1.0,
        log_every=500,
    )
    trainer = Trainer(model, dataset, objective, args, post_step_hook=_post_step_hook)
    return trainer.train()


def evaluate_mante(model, dataset):
    """Compute classification accuracy on the decision period.

    Output channel 0 is high for correct_choice == 1 (right), so we map
    correct_choice=1 to predicted label 0.
    """
    model.eval()
    with torch.no_grad():
        out = model(dataset.inputs)
        y_hat = out.outputs  # (N, T, 2)
    dec_mask = dataset.mask[:, :, 0].bool()
    dec_out = (y_hat * dec_mask.unsqueeze(-1)).sum(dim=1) / dec_mask.sum(dim=1, keepdim=True).clamp_min(1)
    pred_choice = dec_out.argmax(dim=1)
    true_choice = torch.tensor([0 if c['correct_choice'] == 1 else 1 for c in dataset.conditions])
    return (pred_choice == true_choice).float().mean().item()

In [ ]:
# Common hyperparameters
INPUT_DIM = train_ds_mante.input_dim
OUTPUT_DIM = train_ds_mante.output_dim
HIDDEN_DIM = 100
MAX_STEPS = 5000
LR = 1e-3

models_mante = {}

# CTRNN baseline
print("\n=== Training CTRNN baseline ===")
cfg_ctrnn = AutoConfig.for_model(
    'ctrnn',
    input_dim=INPUT_DIM,
    latent_dim=HIDDEN_DIM,
    output_dim=OUTPUT_DIM,
    dt=20.0,
    tau=100.0,
    activation='tanh',
)
models_mante['ctrnn'] = AutoModel.from_config(cfg_ctrnn)
if (MODEL_DIR / 'mante_ctrnn').exists():
    models_mante['ctrnn'] = AutoModel.from_pretrained(MODEL_DIR / 'mante_ctrnn')
    print('Loaded pretrained CTRNN from', MODEL_DIR / 'mante_ctrnn')
else:
    train_mante(models_mante['ctrnn'], train_ds_mante, max_steps=MAX_STEPS, lr=LR)

# SparseRNN variants
for sparsity in [0.05, 0.10, 0.50]:
    print(f"\n=== Training SparseRNN (sparsity={sparsity:.2f}) ===")
    save_path = MODEL_DIR / f'mante_sparse_{int(sparsity*100)}'
    if save_path.exists():
        models_mante[f'sparse_{int(sparsity*100)}'] = AutoModel.from_pretrained(save_path)
        print(f'Loaded pretrained SparseRNN from {save_path}')
        continue
    cfg = AutoConfig.for_model(
        'sparse_rnn',
        input_dim=INPUT_DIM,
        latent_dim=HIDDEN_DIM,
        output_dim=OUTPUT_DIM,
        dt=20.0,
        tau=100.0,
        activation='tanh',
        sparsity=sparsity,
        allow_self_connections=False,
        seed=42,
    )
    models_mante[f'sparse_{int(sparsity*100)}'] = AutoModel.from_config(cfg)
    train_mante(models_mante[f'sparse_{int(sparsity*100)}'], train_ds_mante, max_steps=MAX_STEPS, lr=LR)

# Save models
for name, model in models_mante.items():
    save_path = MODEL_DIR / f'mante_{name}'
    model.save_pretrained(save_path)
    print(f'Saved {name} to {save_path}')

In [ ]:
# Evaluate on held-out test set
results_mante = {}
for name, model in models_mante.items():
    acc = evaluate_mante(model, test_ds_mante)
    results_mante[name] = acc
    print(f"{name:12s}: test accuracy = {acc:.4f}")

# Bar plot
fig, ax = plt.subplots(figsize=(6, 4))
names = list(results_mante.keys())
accuracies = [results_mante[n] for n in names]
colors = ['gray'] + ['#1f77b4', '#ff7f0e', '#2ca02c']
ax.bar(names, accuracies, color=colors)
ax.set_ylabel('Test accuracy')
ax.set_ylim([0, 1])
ax.set_title('Mante task: CTRNN vs. SparseRNN')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()

In [ ]:
# Visualize effective recurrent connectivity matrices for SparseRNN variants
sparse_models = {k: v for k, v in models_mante.items() if k.startswith('sparse_')}
fig, axes = plt.subplots(2, len(sparse_models), figsize=(10, 6))

for col, (name, model) in enumerate(sparse_models.items()):
    mask = model.rec_mask.cpu().numpy()
    W_eff = (model.h2h.weight.detach() * model.rec_mask).cpu().numpy()

    ax_mask = axes[0, col]
    im = ax_mask.imshow(mask, cmap='Greys', aspect='auto')
    ax_mask.set_title(f'{name}\nrec_mask')
    ax_mask.set_xlabel('from j')
    ax_mask.set_ylabel('to i')
    fig.colorbar(im, ax=ax_mask, fraction=0.046)

    ax_w = axes[1, col]
    vmax = np.abs(W_eff).max()
    im2 = ax_w.imshow(W_eff, cmap='bwr_r', aspect='auto', vmin=-vmax, vmax=vmax)
    ax_w.set_title(f'{name}\nW_eff = W * mask')
    ax_w.set_xlabel('from j')
    ax_w.set_ylabel('to i')
    fig.colorbar(im2, ax=ax_w, fraction=0.046)

    in_degree = mask.sum(axis=0)
    out_degree = mask.sum(axis=1)
    print(f"{name:12s}: density = {mask.mean():.4f}, "
          f"mean in-degree = {in_degree.mean():.1f} ± {in_degree.std():.1f}, "
          f"mean out-degree = {out_degree.mean():.1f} ± {out_degree.std():.1f}")

plt.suptitle('SparseRNN recurrent connectivity matrices', y=1.02)
plt.tight_layout()
plt.show()

### ModularRNN connectivity on Mante

We also demonstrate **ModularRNN**, which partitions hidden units into modules with dense intra-module connections and sparse inter-module connections. Here we use 4 modules of 25 neurons each, full intra-module density, and a 5% inter-module connection probability. The block-diagonal structure of the recurrent mask is visible below.

In [ ]:
# Train a ModularRNN on Mante and evaluate
print("\n=== Training ModularRNN (4 modules, p_inter=0.05) ===")
cfg_modular = AutoConfig.for_model(
    'modular_rnn',
    input_dim=INPUT_DIM,
    latent_dim=HIDDEN_DIM,
    output_dim=OUTPUT_DIM,
    dt=20.0,
    tau=100.0,
    activation='tanh',
    n_modules=4,
    p_inter=0.05,
    intra_density=1.0,
    allow_self_connections=False,
    seed=42,
)
model_modular = AutoModel.from_config(cfg_modular)
save_path = MODEL_DIR / 'mante_modular_4_05'

if save_path.exists():
    model_modular = AutoModel.from_pretrained(save_path)
    print('Loaded pretrained ModularRNN from', save_path)
else:
    train_mante(model_modular, train_ds_mante, max_steps=MAX_STEPS, lr=LR)
    model_modular.save_pretrained(save_path)
    print(f'Saved modular_4_05 to {save_path}')

acc_modular = evaluate_mante(model_modular, test_ds_mante)
print(f"modular_4_05: test accuracy = {acc_modular:.4f}")


In [ ]:
# Visualize ModularRNN recurrent connectivity
mask_mod = model_modular.rec_mask.cpu().numpy()
W_eff_mod = (model_modular.h2h.weight.detach() * model_modular.rec_mask).cpu().numpy()

fig, axes = plt.subplots(1, 2, figsize=(8, 4))
im = axes[0].imshow(mask_mod, cmap='Greys', aspect='auto')
axes[0].set_title('ModularRNN rec_mask (4 modules)')
axes[0].set_xlabel('from j')
axes[0].set_ylabel('to i')
fig.colorbar(im, ax=axes[0], fraction=0.046)

vmax = np.abs(W_eff_mod).max()
im2 = axes[1].imshow(W_eff_mod, cmap='bwr_r', aspect='auto', vmin=-vmax, vmax=vmax)
axes[1].set_title('ModularRNN W_eff = W * mask')
axes[1].set_xlabel('from j')
axes[1].set_ylabel('to i')
fig.colorbar(im2, ax=axes[1], fraction=0.046)

# Mark module boundaries
module_size = HIDDEN_DIM // 4
for ax in axes:
    for k in range(1, 4):
        ax.axvline(k * module_size - 0.5, color='cyan', lw=1)
        ax.axhline(k * module_size - 0.5, color='cyan', lw=1)

plt.tight_layout()
plt.show()

in_degree = mask_mod.sum(axis=0)
out_degree = mask_mod.sum(axis=1)
print(f"modular_4_05: density = {mask_mod.mean():.4f}, "
      f"mean in-degree = {in_degree.mean():.1f} ± {in_degree.std():.1f}, "
      f"mean out-degree = {out_degree.mean():.1f} ± {out_degree.std():.1f}")

In [ ]:
def collect_states(model, dataset):
    """Return hidden states (N, T, M) for the dataset."""
    model.eval()
    with torch.no_grad():
        out = model(dataset.inputs)
    return out.states.numpy()


def plot_mante_pca(states, df, pca=None, title='', ax=None):
    """Plot condition-averaged trajectories in PCA space.

    states: (N, T, M)
    df: DataFrame with 'context', 'motion_coh', 'color_coh', 'correct_choice'
    """
    if ax is None:
        fig, ax = plt.subplots(figsize=(4, 4))
    else:
        fig = ax.figure

    N, T, M = states.shape
    flat_states = states.reshape(-1, M)
    if pca is None:
        pca = fit_pca(flat_states, n_components=2)

    colors = np.array([[27, 158, 119], [117, 112, 179]]) / 255.0  # green=motion, purple=color

    for ctx_idx, context in enumerate(['motion', 'color']):
        subset = df[df['context'] == context]
        # Average across coherence conditions for the selected context
        avg = states[subset.index].mean(axis=0)
        avg_pc = pca.transform(avg)
        ax.plot(avg_pc[:, 0], avg_pc[:, 1], 'o-', color=colors[ctx_idx],
                ms=3, lw=1, label=context)
        ax.plot(avg_pc[0, 0], avg_pc[0, 1], '^', color=colors[ctx_idx], ms=6)

    ax.set_xlabel('PC 1')
    ax.set_ylabel('PC 2')
    ax.set_title(title)
    ax.legend(fontsize=7)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    return fig, ax, pca


# Fit PCA on CTRNN states; reuse for sparse networks
states_ctrnn_mante = collect_states(models_mante['ctrnn'], test_ds_mante)
pca_mante = None

fig, axes = plt.subplots(1, 4, figsize=(14, 3))
df_test_mante = pd.DataFrame(test_ds_mante.conditions)
for ax, name in zip(axes, models_mante.keys()):
    states = collect_states(models_mante[name], test_ds_mante)
    _, _, pca_mante = plot_mante_pca(states, df_test_mante, pca=pca_mante, title=name, ax=ax)
plt.suptitle('Mante hidden-state PCA (context-averaged trajectories)', y=1.02)
plt.tight_layout()
plt.show()

## Part III: SparseRNN vs. CTRNN on Parametric Working Memory (DelayComparison)

We repeat the sparse-vs-dense comparison on the DelayComparison task (parametric working memory). The network must compare two stimulus values ($v_1$, $v_2$) separated by a variable delay and report which was larger. This requires a **line attractor** — a continuous manifold of approximately stable fixed points along which the memory of $v_1$ is maintained during the delay.

### Task

Inputs are two channels representing the two stimuli. The target is a binary choice at the end of the trial. We use the NeuroGym `DelayComparison-v0` task via `load_dataset('delay_comparison', ...)`.

### Analysis

After training, we collect activity during a fixed long delay (2000 ms) and project it onto its first two principal components. We then search for fixed points under a zero-coherence input ($v_1 = v_2$) and linearize around a central fixed point to reveal the line-attractor direction.

In [ ]:
# Load DelayComparison for training (variable delay) and analysis (fixed long delay)
timing_train = {'delay': ('choice', [200, 400, 800, 1600, 3200]),
                'response': ('constant', 500)}
ds_pw_train = load_dataset('delay_comparison', timing=timing_train,
                           batch_size=16, seq_len=100, dt=100)
print('Train input dim:', ds_pw_train.input_dim, 'output dim:', ds_pw_train.output_dim)

timing_analysis = {'delay': ('constant', 2000), 'response': ('constant', 500)}
ds_pw_analysis = load_dataset('delay_comparison', timing=timing_analysis,
                              batch_size=16, n_trials=200, dt=100)

# Visualize sample trials (trial-aligned dataset interface)
n_show = 2
ds_viz = load_dataset('delay_comparison', timing=timing_train, batch_size=16,
                      n_trials=n_show, dt=100)
fig, axes = plt.subplots(ds_pw_train.input_dim + 1, 1, figsize=(8, 4), sharex=True)
colors = plt.cm.Set1(np.linspace(0, 1, n_show))
trial_labels = []
for i in range(n_show):
    cond = ds_viz.conditions[i]
    n = cond['n_steps']
    ob = ds_viz.inputs[i, :n].numpy()
    gt = ds_viz.targets[i, :n].numpy()
    t = np.arange(n) * ds_viz.dt / 1000
    for ch in range(ds_pw_train.input_dim):
        axes[ch].plot(t, ob[:, ch], color=colors[i], alpha=0.8, lw=1.2)
    axes[-1].plot(t, gt, color=colors[i], alpha=0.8, lw=1.2)
    trial_labels.append(f'trial {i}: v1={cond.get("v1", "?")}, v2={cond.get("v2", "?")}')

for ch in range(ds_pw_train.input_dim):
    axes[ch].set_ylabel(f'Stim {ch}')
axes[-1].set_ylabel('Target')
axes[-1].set_xlabel('Time (s)')
fig.legend(trial_labels, loc='upper right', fontsize=8)
fig.suptitle('Parametric Working Memory: DelayComparison task', fontsize=10)
plt.tight_layout()
plt.show()

In [ ]:
# Train CTRNN and SparseRNNs on DelayComparison
models_pw = {}

print("\n=== Training CTRNN baseline ===")
cfg_pw_ctrnn = AutoConfig.for_model(
    'ctrnn',
    input_dim=ds_pw_train.input_dim,
    latent_dim=64,
    output_dim=ds_pw_train.output_dim,
    dt=100,
    tau=100,
    activation='relu',
)
models_pw['ctrnn'] = AutoModel.from_config(cfg_pw_ctrnn)
Trainer(models_pw['ctrnn'], ds_pw_train, SupervisedObjective(task_type='classification'),
        TrainingArguments(max_steps=2000, learning_rate=1e-3, log_every=200)).train()

for sparsity in [0.05, 0.10, 0.50]:
    print(f"\n=== Training SparseRNN (sparsity={sparsity:.2f}) ===")
    cfg = AutoConfig.for_model(
        'sparse_rnn',
        input_dim=ds_pw_train.input_dim,
        latent_dim=64,
        output_dim=ds_pw_train.output_dim,
        dt=100,
        tau=100,
        activation='relu',
        sparsity=sparsity,
        allow_self_connections=False,
        seed=42,
    )
    models_pw[f'sparse_{int(sparsity*100)}'] = AutoModel.from_config(cfg)
    Trainer(models_pw[f'sparse_{int(sparsity*100)}'], ds_pw_train,
            SupervisedObjective(task_type='classification'),
            TrainingArguments(max_steps=2000, learning_rate=1e-3, log_every=200)).train()

# Save
for name, model in models_pw.items():
    save_path = MODEL_DIR / f'delaycmp_{name}'
    model.save_pretrained(save_path)
    print(f'Saved {name} to {save_path}')

In [ ]:
# Evaluate on the trial-aligned analysis dataset
def evaluate_delay_comparison(model, dataset):
    '''Classification accuracy on DelayComparison (trial-aligned dataset).'''
    model.eval()
    with torch.no_grad():
        out = model(dataset.inputs)   # batched forward over all trials
    logits_all = out.outputs.numpy()
    correct, total = 0, 0
    for i, cond in enumerate(dataset.conditions):
        pred = int(logits_all[i, cond['n_steps'] - 1].argmax())  # neurogym action index
        correct += (pred == cond['ground_truth'])
        total += 1
    return correct / total

In [ ]:
results_pw = {}
for name, model in models_pw.items():
    acc = evaluate_delay_comparison(model, ds_pw_analysis)
    results_pw[name] = acc
    print(f"{name:12s}: test accuracy = {acc:.4f}")

fig, ax = plt.subplots(figsize=(6, 4))
names = list(results_pw.keys())
accuracies = [results_pw[n] for n in names]
colors = ['gray'] + ['#1f77b4', '#ff7f0e', '#2ca02c']
ax.bar(names, accuracies, color=colors)
ax.set_ylabel('Test accuracy')
ax.set_ylim([0, 1])
ax.set_title('DelayComparison: CTRNN vs. SparseRNN')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()

In [ ]:
# Collect delay-period activity for PCA (from the trial-aligned analysis dataset)
num_trial_pw = 100
activity_dict_pw = {}
trial_infos_pw = {}

model_pw = models_pw['ctrnn']
model_pw.eval()
with torch.no_grad():
    out = model_pw(ds_pw_analysis.inputs)   # batched forward over all trials
states_all = out.states.numpy()

for i, cond in enumerate(ds_pw_analysis.conditions[:num_trial_pw]):
    # Extract activity during delay period only (epoch bounds from conditions)
    s, e = cond['epochs']['delay']
    activity_dict_pw[i] = states_all[i, s:e]
    trial_infos_pw[i] = cond

activity_pw = np.concatenate([activity_dict_pw[i] for i in range(num_trial_pw)], axis=0)
pca_pw = fit_pca(activity_pw, n_components=2)
print('Explained variance ratio:', np.round(pca_pw.explained_variance_ratio, 3))

# Plot CTRNN delay-period trajectories colored by ground truth
fig, axes = plt.subplots(1, 2, sharey=True, sharex=True, figsize=(5, 2.5))
for i in range(num_trial_pw):
    trial = trial_infos_pw[i]
    activity_pc = pca_pw.transform(activity_dict_pw[i])
    color = 'red' if trial['ground_truth'] == 1 else 'blue'
    axes[0].plot(activity_pc[:, 0], activity_pc[:, 1], 'o-', color=color,
                 markersize=1, lw=0.5, alpha=0.3)
    if i < 1:
        axes[1].plot(activity_pc[:, 0], activity_pc[:, 1], 'o-', color=color,
                     markersize=2, lw=1)
axes[0].set_xlabel('PC 1')
axes[0].set_ylabel('PC 2')
axes[0].set_title('All trials')
axes[1].set_title('Single trial')
plt.suptitle('CTRNN delay-period PCA')
plt.tight_layout()
plt.show()

In [ ]:
# Fixed-point search under zero-coherence input (v1 = v2)
task_input_pw = torch.tensor([1, 0], dtype=torch.float32)

fps_pw = find_fixed_points(model_pw, backend='numeric', task_input=task_input_pw,
                           n_candidates=128, n_iters=10000, speed_tol=0.5)
print(f'Found {len(fps_pw)} fixed points')

coords_pw = pca_pw.transform(fps_pw.coords()) if len(fps_pw) else np.empty((0, 2))

fig, ax = plt.subplots(figsize=(4, 4))
for i in range(min(20, num_trial_pw)):
    activity_pc = pca_pw.transform(activity_dict_pw[i])
    trial = trial_infos_pw[i]
    color = 'red' if trial['ground_truth'] == 0 else 'blue'
    ax.plot(activity_pc[:, 0], activity_pc[:, 1], 'o-', color=color, alpha=0.1,
            markersize=1, lw=0.5)

if len(fps_pw):
    ax.plot(coords_pw[:, 0], coords_pw[:, 1], 'x', ms=5, color='black', label='Fixed points')

ax.set_xlabel('PC 1')
ax.set_ylabel('PC 2')
ax.set_title('Fixed points in delay-period PCA space')
ax.legend(fontsize=8)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.show()

In [ ]:
# Jacobian analysis: line attractor direction
if len(fps_pw) > 0:
    fp_coords = fps_pw.coords()
    i_fp = np.argsort(fp_coords[:, 0])[len(fp_coords) // 2]

    lin = linearize(model_pw, fps_pw.points[i_fp].z, task_input=task_input_pw)
    d = dominant_direction(lin)

    fp_z = fps_pw.points[i_fp].z
    end_pts = np.array([fp_z + 2 * d, fp_z - 2 * d])
    end_pts_pc = pca_pw.transform(end_pts)

    fig, ax = plt.subplots(figsize=(4, 4))
    for i in range(min(20, num_trial_pw)):
        activity_pc = pca_pw.transform(activity_dict_pw[i])
        trial = trial_infos_pw[i]
        color = 'red' if trial['ground_truth'] == 0 else 'blue'
        ax.plot(activity_pc[:, 0], activity_pc[:, 1], 'o-', color=color, alpha=0.1,
                markersize=1, lw=0.5)

    ax.plot(coords_pw[:, 0], coords_pw[:, 1], 'x', ms=5, color='black', alpha=0.3)
    ax.plot(coords_pw[i_fp, 0], coords_pw[i_fp, 1], 'x', ms=8, color='red', mew=2,
            label='Selected FP')
    ax.plot(end_pts_pc[:, 0], end_pts_pc[:, 1], '-', color='red', lw=2,
            label='Line attractor')

    ax.set_xlabel('PC 1')
    ax.set_ylabel('PC 2')
    ax.set_title('Line attractor revealed by Jacobian analysis')
    ax.legend(fontsize=8)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    plt.tight_layout()
    plt.show()

    eigvals = lin.eigenvalues
    sorted_idx = np.argsort(-np.abs(eigvals))
    print('Eigenvalues at selected FP (top 5 by magnitude):')
    for idx in sorted_idx[:5]:
        print(f'  λ = {eigvals[idx]:.4f}, |λ| = {np.abs(eigvals[idx]):.4f}')
else:
    print('No fixed points found.')

## Summary

- **seRNN**: The spatial-embedding regularizer biases recurrent connectivity toward short-range, topologically central connections. Training with the NeuralRNN `Trainer` and `ConstrainedSupervisedObjective` reproduces the demo behaviour and structural metrics.
- **Mante**: Both the baseline CTRNN and the SparseRNNs reach high accuracy. Performance degrades gracefully as recurrent connectivity becomes sparser, with 50% sparse networks already approaching the dense baseline. ModularRNN with dense intra-module and sparse inter-module connections also performs well, and its block-structured connectivity matrix is clearly visible.
- **DelayComparison**: SparseRNNs can also support parametric working memory. The delay-period PCA and fixed-point analysis reveal a line attractor in the CTRNN; the sparse networks develop similar geometries as long as enough recurrent connectivity remains.